# Cat → dog transport in DINO feature space

The implementation now lives in `src/brown_bridge/`; this notebook is only an experiment driver. See `docs/mathematical_formulation.md` for the distinction between barycentric summaries, full-coupling displacement paths, OT-CFM, and Schrödinger bridges.

In [ ]:
from pathlib import Path
import torch

from brown_bridge.config import ExperimentConfig, FlowConfig, OTConfig
from brown_bridge.visualization import fit_pca_2d, project_2d
from experiments.cat_dog import run

In [ ]:
config = ExperimentConfig(
    data_root=Path('data'),
    output_dir=Path('outputs/cat_dog'),
    ot=OTConfig(epsilon=0.05, cost='sqeuclidean'),
    flow=FlowConfig(
        macro_steps=10,
        microsteps_per_macro=4,
        particles=1024,
        geometry='ambient',
        bridge_diffusivity=0.0125,  # epsilon / 4 for dX=sqrt(2*tau)dW
    ),
)
summary = run(config)
summary

`transport.pt` contains `n*T + 1` feature states. The particles use endpoint pairs sampled from the complete Sinkhorn plan, so the terminal population approximates the dog marginal instead of the barycentric contraction used by the original notebook.

In [ ]:
artifact = torch.load(config.output_dir / 'transport.pt', weights_only=True)
times = artifact['times']
trajectories = artifact['trajectories']
center, components = fit_pca_2d(
    artifact['cat_features'], artifact['dog_features'], trajectories.reshape(-1, trajectories.shape[-1])
)
projected_trajectories = project_2d(trajectories, center, components)
times.shape, trajectories.shape, projected_trajectories.shape